In [1]:
import xarray as xr

In [2]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/OM4_Horizontal_Regrid_Old.zarr"
)

# data = xr.open_zarr(
#     "/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr"
# )

# data = xr.open_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/test_CMIP6_GFDL-CM4.piControl.r1i1p1f1.zarr")

In [3]:
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

### Convert and save 3D data for training

Make sure you are using atleast 10 cores!

In [4]:
from dask.diagnostics import ProgressBar

In [11]:
import numpy as np
def manual_v0_fixes(ds_input: xr.Dataset) -> xr.Dataset:
    """Manual fixes for the already existing data (for now only v0.0). This should not be used in the future"""
    # fixes that should be checked and fixes on the input data
    # area = xr.open_dataset(
    #     "gs://leap-persistent/sd5313/grids_CM2x.zarr", engine="zarr", chunks={}
    # )["area_C"].rename({"xu_ocean": "x", "yu_ocean": "y"})
    # print("area", area)
    # area = xr.open_dataset("/pscratch/sd/s/suryad/data/Grid_New.nc")["area_C"].rename({"xu_ocean": "x", "yu_ocean": "y"})
    # print("area", area)
    # from https://github.com/m2lines/ocean_emulators/issues/17
    dz = xr.DataArray(
        [
            5,
            10,
            15,
            20,
            30,
            50,
            70,
            100,
            150,
            200,
            250,
            300,
            400,
            500,
            600,
            800,
            1000,
            1000,
            1000,
        ],
        dims=["lev"],
    )
    z = xr.DataArray(
        [
            2.5,
            10,
            22.5,
            40,
            65,
            105,
            165,
            250,
            375,
            550,
            775,
            1050,
            1400,
            1850,
            2400,
            3100,
            4000,
            5000,
            6000,
        ],
        dims="lev",
    )
    wetmask = ~np.isnan(ds_input.thetao.isel(time=0).reset_coords(drop=True)).load()
    # lon = xr.ones_like(ds_input.y) * ds_input.x
    # lat = ds_input.y * xr.ones_like(ds_input.x)
    # ds_input = ds_input.assign_coords(
    #     areacello=area, dz=dz, lev=z, wetmask=wetmask, lon=lon, lat=lat
    # )
    ds_input = ds_input.assign_coords(
        dz=dz, lev=z, wetmask=wetmask
    )
    # give a dummy commit hash
    ds_input.attrs["m2lines/ocean-emulators_git_hash"] = "dummy"
    return ds_input

data = manual_v0_fixes(data)

In [12]:
ds = data
ds

<xarray.Dataset>
Dimensions:            (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time               (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x                  (x) float64 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * y                  (y) float64 -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
    dz                 (lev) int64 5 10 15 20 30 50 ... 600 800 1000 1000 1000
  * lev                (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    wetmask            (lev, y, x) bool False False False ... False False False
Data variables: (12/84)
    hfds               (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so                 (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo              (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo              (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao             (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo                 (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    ...                 ...
    uo_lev_5000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_5000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_6000_0  (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    uo_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  dummy

In [13]:
ds["lev"]

<xarray.DataArray 'lev' (lev: 19)>
array([2.50e+00, 1.00e+01, 2.25e+01, 4.00e+01, 6.50e+01, 1.05e+02, 1.65e+02,
       2.50e+02, 3.75e+02, 5.50e+02, 7.75e+02, 1.05e+03, 1.40e+03, 1.85e+03,
       2.40e+03, 3.10e+03, 4.00e+03, 5.00e+03, 6.00e+03])
Coordinates:
    dz       (lev) int64 5 10 15 20 30 50 70 ... 400 500 600 800 1000 1000 1000
  * lev      (lev) float64 2.5 10.0 22.5 40.0 65.0 ... 3.1e+03 4e+03 5e+03 6e+03

In [14]:
assert [str(lev).replace(".", "_") for lev in ds["lev"].values] == ['2_5', '10_0', '22_5', '40_0', '65_0', '105_0', '165_0', '250_0', '375_0', '550_0', '775_0', '1050_0', '1400_0', '1850_0', '2400_0', '3100_0', '4000_0', '5000_0', '6000_0']

In [15]:
for lev in ds["lev"].values:
    lev_str = str(lev).replace(".", "_")

    # Create new variables for each original variable with the lev dimension
    ds[f"vo_lev_{lev_str}"] = ds["vo"].sel(lev=lev)
    ds[f"thetao_lev_{lev_str}"] = ds["thetao"].sel(lev=lev)
    ds[f"uo_lev_{lev_str}"] = ds["uo"].sel(lev=lev)
    ds[f"so_lev_{lev_str}"] = ds["so"].sel(lev=lev)

ds = ds.drop_vars(["vo", "thetao", "uo", "so"])
ds

<xarray.Dataset>
Dimensions:            (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time               (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x                  (x) float64 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * y                  (y) float64 -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
    dz                 (lev) int64 5 10 15 20 30 50 ... 600 800 1000 1000 1000
  * lev                (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    wetmask            (lev, y, x) bool False False False ... False False False
Data variables: (12/80)
    hfds               (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauuo              (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo              (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos                (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_2_5         (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_2_5     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...                 ...
    uo_lev_5000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_5000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_6000_0  (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    uo_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  dummy

In [16]:
with ProgressBar():
    ds_mean = ds.mean().compute()

[########################################] | 100% Completed | 519.43 s


In [17]:
ds_mean.to_zarr("/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0_means")

In [18]:
with ProgressBar():
    ds_std = ds.std().compute()

[########################################] | 100% Completed | 11m 40s


In [19]:
ds_std.to_zarr("/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0_stds")

In [20]:
with ProgressBar():
    ds.to_zarr("/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0")

[########################################] | 100% Completed | 526.86 s


In [26]:
ds

<xarray.Dataset>
Dimensions:            (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time               (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x                  (x) float64 0.5 1.5 2.5 3.5 ... 356.5 357.5 358.5 359.5
  * y                  (y) float64 -89.5 -88.5 -87.5 -86.5 ... 87.5 88.5 89.5
    dz                 (lev) int64 5 10 15 20 30 50 ... 600 800 1000 1000 1000
  * lev                (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    wetmask            (lev, y, x) bool False False False ... False False False
Data variables: (12/80)
    hfds               (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauuo              (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo              (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos                (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_2_5         (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_2_5     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...                 ...
    uo_lev_5000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_5000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao_lev_6000_0  (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    uo_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_6000_0      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  dummy

#### Wet mask

In [27]:
import xarray as xr

In [49]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/OM4_Horizontal_Regrid_Old.zarr"
)

import numpy as np
def manual_v0_fixes(ds_input: xr.Dataset) -> xr.Dataset:
    """Manual fixes for the already existing data (for now only v0.0). This should not be used in the future"""
    # fixes that should be checked and fixes on the input data
    # area = xr.open_dataset(
    #     "gs://leap-persistent/sd5313/grids_CM2x.zarr", engine="zarr", chunks={}
    # )["area_C"].rename({"xu_ocean": "x", "yu_ocean": "y"})
    # print("area", area)
    # area = xr.open_dataset("/pscratch/sd/s/suryad/data/Grid_New.nc")["area_C"].rename({"xu_ocean": "x", "yu_ocean": "y"})
    # print("area", area)
    # from https://github.com/m2lines/ocean_emulators/issues/17
    dz = xr.DataArray(
        [
            5,
            10,
            15,
            20,
            30,
            50,
            70,
            100,
            150,
            200,
            250,
            300,
            400,
            500,
            600,
            800,
            1000,
            1000,
            1000,
        ],
        dims=["lev"],
    )
    z = xr.DataArray(
        [
            2.5,
            10,
            22.5,
            40,
            65,
            105,
            165,
            250,
            375,
            550,
            775,
            1050,
            1400,
            1850,
            2400,
            3100,
            4000,
            5000,
            6000,
        ],
        dims="lev",
    )
    wetmask = ~np.isnan(ds_input.thetao.isel(time=0).reset_coords(drop=True)).load()
    # lon = xr.ones_like(ds_input.y) * ds_input.x
    # lat = ds_input.y * xr.ones_like(ds_input.x)
    # ds_input = ds_input.assign_coords(
    #     areacello=area, dz=dz, lev=z, wetmask=wetmask, lon=lon, lat=lat
    # )
    ds_input = ds_input.assign_coords(
        dz=dz, lev=z, wetmask=wetmask
    )
    # give a dummy commit hash
    ds_input.attrs["m2lines/ocean-emulators_git_hash"] = "dummy"
    return ds_input

data = manual_v0_fixes(data)

levels = 19
# data = xr.open_zarr(
#     "/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr"
# )
# data = xr.open_zarr(
#     "/vast/sd5313/data/m2lines/3D_ocean_data/test_CMIP6_GFDL-CM4.piControl.r1i1p1f1.zarr"
# )
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
    dz       (lev) int64 5 10 15 20 30 50 70 ... 400 500 600 800 1000 1000 1000
  * lev      (lev) float64 2.5 10.0 22.5 40.0 65.0 ... 3.1e+03 4e+03 5e+03 6e+03
    wetmask  (lev, y, x) bool False False False False ... False False False
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  dummy

In [50]:
data = data.drop(["tauuo", "tauvo", "hfds"])
# data = data.drop(["tauuo", "tauvo", "hft"])
data

<xarray.Dataset>
Dimensions:  (time: 4745, lev: 19, y: 180, x: 360)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
    dz       (lev) int64 5 10 15 20 30 50 70 ... 400 500 600 800 1000 1000 1000
  * lev      (lev) float64 2.5 10.0 22.5 40.0 65.0 ... 3.1e+03 4e+03 5e+03 6e+03
    wetmask  (lev, y, x) bool False False False False ... False False False
Data variables:
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  dummy

In [51]:
import numpy as np
import torch


def get_wet_mask(inputs, device="cpu"):
    wet = xr.zeros_like(inputs[0][0])
    # inputs[0][0,12,12] = np.nan
    for data in inputs:
        wet += np.isnan(data[0])

    wet_nan = xr.where(wet != 0, np.nan, 1).to_numpy()
    wet = np.isnan(xr.where(wet == 0, np.nan, 0))
    wet = np.nan_to_num(wet.to_numpy())
    wet = torch.from_numpy(wet).type(torch.float32).to(device=device)
    return wet, wet_nan

In [52]:
data.lev.values

array([2.50e+00, 1.00e+01, 2.25e+01, 4.00e+01, 6.50e+01, 1.05e+02,
       1.65e+02, 2.50e+02, 3.75e+02, 5.50e+02, 7.75e+02, 1.05e+03,
       1.40e+03, 1.85e+03, 2.40e+03, 3.10e+03, 4.00e+03, 5.00e+03,
       6.00e+03])

In [53]:
wet_stacked = []
for i, lev in enumerate(ds["lev"].values):
    inputs = []
    inputs.append(data["uo"].sel(lev=lev))
    inputs.append(data["vo"].sel(lev=lev))
    inputs.append(data["thetao"].sel(lev=lev))
    inputs.append(data["so"].sel(lev=lev))
    if i == 0:
        inputs.append(data["zos"])

    inputs = tuple(inputs)
    wet, _ = get_wet_mask(inputs)
    wet_stacked.append(wet)

In [54]:
wet_3D = torch.stack(wet_stacked)
wet_3D.shape

torch.Size([19, 180, 360])

In [55]:
DEPTH_LEVELS = ['2_5',
 '10_0',
 '22_5',
 '40_0',
 '65_0',
 '105_0',
 '165_0',
 '250_0',
 '375_0',
 '550_0',
 '775_0',
 '1050_0',
 '1400_0',
 '1850_0',
 '2400_0',
 '3100_0',
 '4000_0',
 '5000_0',
 '6000_0']

INPT_VARS = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "ur", "vr"],
    "3": ["um", "vm", "Tm"],
    "4": ["um", "vm", "ur", "vr", "Tm", "Tr"],
    "5": ["ur", "vr"],
    "6": ["ur", "vr", "Tr"],
    "7": ["Tm"],
    "8": ["Tm", "Tr"],
    "9": ["u", "v"],
    "10": ["u", "v", "T"],
    "11": ["tau_u", "tau_v"],
    "12": ["tau_u", "tau_v", "t_ref"],
    "3D": ["uo", "vo", "thetao", "so", "zos"],
    "3D_5": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS[:5]
    ]
    + ["zos"],
    "3D_all": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS
    ]
    + ["zos"],
    "3D_SST_all": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS if not (k == "thetao_lev_" and j == DEPTH_LEVELS[0]) 
    ] 
    + ["zos"],
}

In [71]:
out_list = INPT_VARS["3D_all"]
lev_map = {str(lev).replace('.', '_'): i for i, lev in enumerate(ds["lev"].values)}
# print(out_list[:38] + out_list[39:])

In [72]:
out_list[0].split("lev_")[-1]

'2_5'

In [76]:
final_wet = []
for var in out_list:
    try:
        level = lev_map[var.split("lev_")[-1]]
        print(var, level)
    except Exception as e:
        level = 0
    final_wet.append(wet_3D[level])

uo_lev_2_5 0
uo_lev_10_0 1
uo_lev_22_5 2
uo_lev_40_0 3
uo_lev_65_0 4
uo_lev_105_0 5
uo_lev_165_0 6
uo_lev_250_0 7
uo_lev_375_0 8
uo_lev_550_0 9
uo_lev_775_0 10
uo_lev_1050_0 11
uo_lev_1400_0 12
uo_lev_1850_0 13
uo_lev_2400_0 14
uo_lev_3100_0 15
uo_lev_4000_0 16
uo_lev_5000_0 17
uo_lev_6000_0 18
vo_lev_2_5 0
vo_lev_10_0 1
vo_lev_22_5 2
vo_lev_40_0 3
vo_lev_65_0 4
vo_lev_105_0 5
vo_lev_165_0 6
vo_lev_250_0 7
vo_lev_375_0 8
vo_lev_550_0 9
vo_lev_775_0 10
vo_lev_1050_0 11
vo_lev_1400_0 12
vo_lev_1850_0 13
vo_lev_2400_0 14
vo_lev_3100_0 15
vo_lev_4000_0 16
vo_lev_5000_0 17
vo_lev_6000_0 18
thetao_lev_2_5 0
thetao_lev_10_0 1
thetao_lev_22_5 2
thetao_lev_40_0 3
thetao_lev_65_0 4
thetao_lev_105_0 5
thetao_lev_165_0 6
thetao_lev_250_0 7
thetao_lev_375_0 8
thetao_lev_550_0 9
thetao_lev_775_0 10
thetao_lev_1050_0 11
thetao_lev_1400_0 12
thetao_lev_1850_0 13
thetao_lev_2400_0 14
thetao_lev_3100_0 15
thetao_lev_4000_0 16
thetao_lev_5000_0 17
thetao_lev_6000_0 18
so_lev_2_5 0
so_lev_10_0 1
so_lev_22

In [77]:
wet = torch.stack(final_wet)
wet.shape

torch.Size([77, 180, 360])

In [78]:
assert wet.shape[0] == (levels * 4 + 1)

In [79]:
# wet = torch.cat([wet[:38], wet[39:]], axis=0)
# wet.shape

In [80]:
torch.save(wet, "/pscratch/sd/s/suryad/data/3D_wet_OM4_v0.0.pt")

### Convert and save surface data

Make sure you are using atleast 8 cores!

In [81]:
import sys

sys.path.append("../src/")

In [82]:
from utils.data_utils import get_wet_mask

In [83]:
inputs = []
inputs.append(data["uo"].sel(lev=2.5))
inputs.append(data["vo"].sel(lev=2.5))
inputs.append(data["thetao"].sel(lev=2.5))
inputs.append(data["so"].sel(lev=2.5))
inputs.append(data["zos"])
inputs = tuple(inputs)
wet, _ = get_wet_mask(inputs)
print("Wet resolution:", wet.shape)

Wet resolution: torch.Size([180, 360])


In [84]:
# print("Calculating mask tensors")
# wet, wet_nan = get_wet_mask(inputs, "cpu")
# # wet_bool = np.array(wet.cpu()).astype(bool)
# # wet_lap = compute_laplacian_wet(wet_nan, 4)  # hardcoded
# # wet_lap = xr.where(wet_lap == 0, 1, np.nan)
# # wet_lap = np.nan_to_num(wet_lap)
# print("Wet resolution:", wet.shape)

In [85]:
import torch

In [86]:
torch.save(wet, "/pscratch/sd/s/suryad/data/surface_wet_OM4_v0.0.pt")

#### Extra for surface training

In [ ]:
from dask.diagnostics import ProgressBar

In [ ]:
with ProgressBar():
    data_mean = data.mean().compute()

[########################################] | 100% Completed | 259.26 s


In [ ]:
data_mean.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_means")

In [ ]:
with ProgressBar():
    data_std = data.std().compute()

[########################################] | 100% Completed | 244.16 s


In [ ]:
data_std.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data_stds")

In [ ]:
data.to_zarr("/vast/sd5313/data/m2lines/3D_ocean_data/surface_data")

### Test

In [26]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/OM4_5daily_v0.2.1.zarr"
)
data


<xarray.Dataset>
Dimensions:         (y: 180, x: 360, lev: 19, time: 4745, y_b: 181, x_b: 361)
Coordinates:
    areacello       (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    dz              (lev) int64 dask.array<chunksize=(19,), meta=np.ndarray>
    lat             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lat_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
  * lev             (lev) float64 2.5 10.0 22.5 40.0 ... 4e+03 5e+03 6e+03
    lon             (y, x) float64 dask.array<chunksize=(90, 360), meta=np.ndarray>
    lon_b           (y_b, x_b) float64 dask.array<chunksize=(91, 361), meta=np.ndarray>
    ocean_fraction  (lev, y, x) float64 dask.array<chunksize=(19, 180, 360), meta=np.ndarray>
  * time            (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
    wetmask         (lev, y, x) bool dask.array<chunksize=(10, 90, 360), meta=np.ndarray>
  * x               (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y               (y) float64 -89.24 -88.25 -87.25 ... 87.25 88.25 89.24
Dimensions without coordinates: y_b, x_b
Data variables:
    hfds            (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo           (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao          (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo              (time, lev, y, x) float32 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos             (time, y, x) float32 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
Attributes:
    m2lines/ocean-emulators_git_hash:  https://github.com/m2lines/ocean_emula...
    regrid_method:                     conservative

In [3]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/OM4_Horizontal_Regrid_Old.zarr"
)
data

<xarray.Dataset>
Dimensions:  (time: 4745, y: 180, x: 360, lev: 19)
Coordinates:
  * time     (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x        (x) float64 0.5 1.5 2.5 3.5 4.5 ... 355.5 356.5 357.5 358.5 359.5
  * y        (y) float64 -89.5 -88.5 -87.5 -86.5 -85.5 ... 86.5 87.5 88.5 89.5
Dimensions without coordinates: lev
Data variables:
    hfds     (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    tauuo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    tauvo    (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    thetao   (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    uo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    vo       (time, lev, y, x) float64 dask.array<chunksize=(1, 19, 180, 360), meta=np.ndarray>
    zos      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [37]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/realv00/3D_data_OM4_v0.0"
)
data

<xarray.Dataset>
Dimensions:        (time: 4745, y: 180, x: 360)
Coordinates:
  * time           (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x              (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y              (y) float64 -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
Data variables: (12/80)
    hfds           (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_0       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_1       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_10      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_11      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_12      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...             ...
    vo_lev_5       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_7       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_8       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_9       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos            (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [24]:
import numpy as np
import torch

# Experiment inputs and outputs
DEPTH_LEVELS = ['2_5',
 '10_0',
 '22_5',
 '40_0',
 '65_0',
 '105_0',
 '165_0',
 '250_0',
 '375_0',
 '550_0',
 '775_0',
 '1050_0',
 '1400_0',
 '1850_0',
 '2400_0',
 '3100_0',
 '4000_0',
 '5000_0',
 '6000_0']

INPT_VARS = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "ur", "vr"],
    "3": ["um", "vm", "Tm"],
    "4": ["um", "vm", "ur", "vr", "Tm", "Tr"],
    "5": ["ur", "vr"],
    "6": ["ur", "vr", "Tr"],
    "7": ["Tm"],
    "8": ["Tm", "Tr"],
    "9": ["u", "v"],
    "10": ["u", "v", "T"],
    "11": ["tau_u", "tau_v"],
    "12": ["tau_u", "tau_v", "t_ref"],
    "3D": ["uo", "vo", "thetao", "so", "zos"],
    "3D_5": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS[:5]
    ]
    + ["zos"],
    "3D_all": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS
    ]
    + ["zos"],
    "3D_SST_all": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS if not (k == "thetao_lev_" and j == DEPTH_LEVELS[0]) 
    ] 
    + ["zos"],
}
EXTRA_VARS = {
    "1": ["ur", "vr"],
    "2": ["ur", "vr", "Tm"],
    "3": ["Tm"],
    "4": ["ur", "vr", "Tm", "Tr"],
    "5": [],
    "6": ["um", "vm"],
    "7": ["um", "vm", "Tm"],
    "8": ["um", "vm", "Tm", "Tr"],
    "9": ["ur", "vr", "tau_u", "tau_v"],
    "10": ["tau_u", "tau_v"],
    "11": ["t_ref"],
    "12": ["tau_u", "tau_v", "t_ref"],
    "13": ["ur", "vr", "Tr", "tau_u", "tau_v", "t_ref"],
    "3D": ["tauuo", "tauvo", "hfds"],
    "3D_5": ["tauuo", "tauvo", "hfds"],
    "3D_all": ["tauuo", "tauvo", "hfds"],
    "3D_SST_all": ["tauuo", "tauvo", "hfds", "thetao_lev_0"],
}
OUT_VARS = {
    "1": ["um", "vm"],
    "2": ["um", "vm", "Tm"],
    "3": ["ur", "vr"],
    "4": ["ur", "vr", "Tr"],
    "5": ["u", "v"],
    "6": ["u", "v", "T"],
    "3D": ["uo", "vo", "thetao", "so", "zos"],
    "3D_5": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS[:5]
    ]
    + ["zos"],
    "3D_all": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS
    ]
    + ["zos"],
    "3D_SST_all": [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in DEPTH_LEVELS if not (k == "thetao_lev_" and j == DEPTH_LEVELS[0]) 
    ] 
    + ["zos"],
}


In [19]:
data

<xarray.Dataset>
Dimensions:        (time: 4745, y: 180, x: 360)
Coordinates:
  * time           (time) object 1958-01-03 12:00:00 ... 2022-12-29 12:00:00
  * x              (x) float64 0.5 1.5 2.5 3.5 4.5 ... 356.5 357.5 358.5 359.5
  * y              (y) float64 -89.5 -88.5 -87.5 -86.5 ... 86.5 87.5 88.5 89.5
Data variables: (12/80)
    hfds           (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_0       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_1       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_10      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_11      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    so_lev_12      (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    ...             ...
    vo_lev_5       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_6       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_7       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_8       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    vo_lev_9       (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>
    zos            (time, y, x) float64 dask.array<chunksize=(1, 180, 360), meta=np.ndarray>

In [53]:
data = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0"
)
# mean and std
data_mean = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0_means"
)
data_std = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0_stds"
)

# Extract single timestep
data_single = data[INPT_VARS["3D_all"]].isel(time=0)

# Normalize
data_norm = (data_single - data_mean[INPT_VARS["3D_all"]]) / data_std[INPT_VARS["3D_all"]]

# Unnormalize
data_unnorm = data_norm * data_std[INPT_VARS["3D_all"]] + data_mean[INPT_VARS["3D_all"]]

# Verify that the unnormalized data is the same as the original data
assert (data_single - data_unnorm).sum() == 0


In [6]:
data2 = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0_means"
)
data2

<xarray.Dataset>
Dimensions:            ()
Data variables: (12/80)
    hfds               float64 ...
    so_lev_1050_0      float64 ...
    so_lev_105_0       float64 ...
    so_lev_10_0        float64 ...
    so_lev_1400_0      float64 ...
    so_lev_165_0       float64 ...
    ...                 ...
    vo_lev_5000_0      float64 ...
    vo_lev_550_0       float64 ...
    vo_lev_6000_0      float64 ...
    vo_lev_65_0        float64 ...
    vo_lev_775_0       float64 ...
    zos                float64 ...

In [52]:
import torch 
d1 = torch.load("/pscratch/sd/s/suryad/data/3D_wet_OM4_v0.0.pt")
d2 = torch.load("/pscratch/sd/s/suryad/data/realv00/3D_wet_OM4_v0.0.pt")
assert (d1 - d2).sum() == 0

d1 = torch.load("/pscratch/sd/s/suryad/data/surface_wet_OM4_v0.0.pt")
d2 = torch.load("/pscratch/sd/s/suryad/data/realv00/surface_wet_OM4_v0.0.pt")
assert (d1 - d2).sum() == 0

In [16]:
# mean and std
data_mean = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0_means"
)
data_std = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/3D_data_OM4_v0.0_stds"
)

data_mean2 = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/realv00/3D_data_OM4_v0.0_means"
)
data_std2 = xr.open_zarr(
    "/pscratch/sd/s/suryad/data/realv00/3D_data_OM4_v0.0_stds"
)

In [17]:
data_mean.values == data_mean2.values

False

In [25]:
data_mean[INPT_VARS["3D_all"]].load()

<xarray.Dataset>
Dimensions:            ()
Data variables: (12/77)
    uo_lev_2_5         float64 0.004764
    uo_lev_10_0        float64 0.0033
    uo_lev_22_5        float64 0.005722
    uo_lev_40_0        float64 0.009696
    uo_lev_65_0        float64 0.01151
    uo_lev_105_0       float64 0.01173
    ...                 ...
    so_lev_2400_0      float64 34.76
    so_lev_3100_0      float64 34.76
    so_lev_4000_0      float64 34.74
    so_lev_5000_0      float64 34.72
    so_lev_6000_0      float64 34.71
    zos                float64 -0.1926

In [26]:
ls = [
        k + str(j)
        for k in ["uo_lev_", "vo_lev_", "thetao_lev_", "so_lev_"]
        for j in range(19)
    ]+ ["zos"]
data_mean2[ls].load()

<xarray.Dataset>
Dimensions:        ()
Data variables: (12/77)
    uo_lev_0       float64 0.004764
    uo_lev_1       float64 0.0033
    uo_lev_2       float64 0.005722
    uo_lev_3       float64 0.009696
    uo_lev_4       float64 0.01151
    uo_lev_5       float64 0.01173
    ...             ...
    so_lev_14      float64 34.76
    so_lev_15      float64 34.76
    so_lev_16      float64 34.74
    so_lev_17      float64 34.72
    so_lev_18      float64 34.71
    zos            float64 -0.1926

In [36]:
assert (data_mean[INPT_VARS["3D_all"]].load().to_array().values == data_mean2[ls].load().to_array().values).all()
assert (data_std[INPT_VARS["3D_all"]].load().to_array().values == data_std2[ls].load().to_array().values).all()